# CatBoost Edge AI Deployment

This notebook prepares the selected CatBoost occupancy classification model for deployment on an edge device (Raspberry Pi).

The model predicts the occupancy level of a room 30 minutes ahead using building sensor data.

In [1]:
from pathlib import Path
import json
import time
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score

from catboost import CatBoostClassifier
import joblib

RANDOM_STATE = 42

DATA_PATH = Path("../data/processed/model_table_5min.csv")
REPORT_DIR = Path("../reports/occupancy_classification")

CLASS_LABELS = [0, 1, 2, 3, 4]
CLASS_NAMES = ["Empty", "Low", "Moderate", "High", "Very high"]

print("Working directory:", Path.cwd())
print("Dataset path:", DATA_PATH.resolve())

Working directory: c:\Users\leena\OneDrive\Desktop\adaptive-district-backbone\notebooks
Dataset path: C:\Users\leena\OneDrive\Desktop\adaptive-district-backbone\data\processed\model_table_5min.csv


In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

data = pd.read_csv(DATA_PATH)

print("Dataset shape:", data.shape)
print("Number of days:", data["DayId"].nunique())
print("Rooms:", sorted(data["room"].dropna().unique()))

display(data.head())

Dataset shape: (36432, 58)
Number of days: 44
Rooms: [np.int64(1), np.int64(2), np.int64(3)]


,DayId,bin,occ,cam1_fw,cam1_bw,cam2_fw,cam2_bw,co2,temperature,humidity,...,humidity_diff,cam1_fw_last15,cam1_bw_last15,cam2_fw_last15,cam2_bw_last15,target,capacity,event_day,occ_ratio,is_lecture
0,0,6,0.0,0.0,0.0,0.0,0.0,482.880005,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1
1,0,7,0.0,0.0,0.0,0.0,0.0,480.000000,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1
2,0,8,0.0,0.0,0.0,0.0,0.0,475.839996,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1
3,0,9,0.0,0.0,0.0,0.0,0.0,478.720001,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1
4,0,10,0.0,0.0,0.0,0.0,0.0,473.920013,23.6,52.941177,...,0.0,0.0,0.0,0.0,0.0,0.0,84,0,0.0,1


In [3]:
occupancy_ratio_future = data["target"] / data["capacity"]

data["target_class"] = pd.cut(
    occupancy_ratio_future,
    bins=[-0.001, 0, 0.25, 0.50, 0.75, np.inf],
    labels=CLASS_LABELS,
    include_lowest=True
).astype("Int64")

data = data.dropna(
    subset=["target_class", "DayId", "capacity"]
).copy()

data["target_class"] = data["target_class"].astype(int)

class_distribution = (
    data["target_class"]
    .value_counts()
    .reindex(CLASS_LABELS, fill_value=0)
)

class_distribution.index = CLASS_NAMES

display(class_distribution.to_frame("Number of rows"))

,Number of rows
Empty,16523
Low,13301
Moderate,3983
High,1883
Very high,742


In [4]:
leakage_or_id_cols = {
    "target", "target_class", "event_day", "DayId",
    "occ_future", "target_future", "future_occupancy"
}

feature_cols = [
    c for c in data.columns
    if c not in leakage_or_id_cols
    and pd.api.types.is_numeric_dtype(data[c])
]

feature_cols = [
    c for c in feature_cols
    if data[c].notna().any()
]

X = data[feature_cols].copy()
y = data["target_class"].astype(int)
groups = data["DayId"]

print("Number of features:", len(feature_cols))
print("\nFeature names:")
print(feature_cols)

print("\nFeature matrix:", X.shape)
print("Target:", y.shape)

Number of features: 55

Feature names:
['bin', 'occ', 'cam1_fw', 'cam1_bw', 'cam2_fw', 'cam2_bw', 'co2', 'temperature', 'humidity', 'illuminance', 'vav', 'co2_age', 'temperature_age', 'humidity_age', 'illuminance_age', 'vav_age', 'room', 'workday', 'month', 'hour_sin', 'hour_cos', 'occ_lag1', 'occ_lag2', 'occ_lag3', 'occ_lag6', 'co2_lag1', 'co2_lag2', 'co2_lag3', 'co2_lag6', 'temperature_lag1', 'temperature_lag2', 'temperature_lag3', 'temperature_lag6', 'humidity_lag1', 'humidity_lag2', 'humidity_lag3', 'humidity_lag6', 'illuminance_lag1', 'illuminance_lag2', 'illuminance_lag3', 'illuminance_lag6', 'vav_lag1', 'vav_lag2', 'vav_lag3', 'vav_lag6', 'co2_diff', 'temperature_diff', 'humidity_diff', 'cam1_fw_last15', 'cam1_bw_last15', 'cam2_fw_last15', 'cam2_bw_last15', 'capacity', 'occ_ratio', 'is_lecture']

Feature matrix: (36432, 55)
Target: (36432,)


In [5]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

assert set(groups_train).isdisjoint(set(groups_test))

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training days:", groups_train.nunique())
print("Testing days:", groups_test.nunique())

Training rows: 28980
Testing rows: 7452
Training days: 35
Testing days: 9


In [6]:
catboost_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    CatBoostClassifier(
        iterations=350,
        depth=5,
        learning_rate=0.04,
        loss_function="MultiClass",
        verbose=False,
        random_seed=RANDOM_STATE,
        auto_class_weights="Balanced"
    )
)

start_time = time.perf_counter()

catboost_pipeline.fit(X_train, y_train)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.2f} seconds")

Training time: 7.64 seconds


In [7]:
predictions = catboost_pipeline.predict(X_test)
predictions = np.asarray(predictions).reshape(-1).astype(int)

print(classification_report(
    y_test,
    predictions,
    labels=CLASS_LABELS,
    target_names=CLASS_NAMES,
    zero_division=0
))

print("Macro F1:", round(
    f1_score(y_test, predictions, average="macro"), 4
))

              precision    recall  f1-score   support

       Empty       0.92      0.91      0.91      3067
         Low       0.87      0.81      0.84      2874
    Moderate       0.60      0.67      0.64       761
        High       0.57      0.70      0.63       550
   Very high       0.59      0.55      0.57       200

    accuracy                           0.82      7452
   macro avg       0.71      0.73      0.72      7452
weighted avg       0.83      0.82      0.83      7452

Macro F1: 0.7172


In [8]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    catboost_pipeline,
    REPORT_DIR / "catboost_pipeline.joblib"
)

with open(REPORT_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=4)

class_metadata = {
    "0": "Empty",
    "1": "Low",
    "2": "Moderate",
    "3": "High",
    "4": "Very high"
}

with open(REPORT_DIR / "class_labels.json", "w") as f:
    json.dump(class_metadata, f, indent=4)

In [9]:
loaded_model = joblib.load(
    REPORT_DIR / "catboost_pipeline.joblib"
)

with open(REPORT_DIR / "feature_columns.json", "r") as f:
    loaded_features = json.load(f)

with open(REPORT_DIR / "class_labels.json", "r") as f:
    loaded_classes = json.load(f)

print("Model loaded successfully")
print("Number of features:", len(loaded_features))
print("Classes:", loaded_classes)

Model loaded successfully
Number of features: 55
Classes: {'0': 'Empty', '1': 'Low', '2': 'Moderate', '3': 'High', '4': 'Very high'}


In [10]:
test_predictions = np.asarray(
    loaded_model.predict(X_test)
).reshape(-1).astype(int)

print(classification_report(
    y_test,
    test_predictions,
    labels=CLASS_LABELS,
    target_names=CLASS_NAMES,
    zero_division=0
))

assert np.array_equal(predictions, test_predictions)

print("Predictions match the original model!")

              precision    recall  f1-score   support

       Empty       0.92      0.91      0.91      3067
         Low       0.87      0.81      0.84      2874
    Moderate       0.60      0.67      0.64       761
        High       0.57      0.70      0.63       550
   Very high       0.59      0.55      0.57       200

    accuracy                           0.82      7452
   macro avg       0.71      0.73      0.72      7452
weighted avg       0.83      0.82      0.83      7452

Predictions match the original model!


In [11]:
import importlib
import edge_inference

importlib.reload(edge_inference)

result = edge_inference.predict_occupancy(sample_input)

print(result)

NameError: name 'sample_input' is not defined

In [ ]:
import sys

sys.path.insert(0, str(Path.cwd().parent))

from edge_inference import predict_occupancy

sample_input = X_test.iloc[0].to_dict()

result = predict_occupancy(sample_input)

print(result)

{'class_id': 0, 'occupancy_level': 'Empty'}


In [ ]:
import time
import numpy as np

sample_input = X_test.iloc[0].to_dict()

# Warm-up
for _ in range(10):
    predict_occupancy(sample_input)

# Benchmark
times = []

for _ in range(1000):
    start = time.perf_counter()
    predict_occupancy(sample_input)
    times.append((time.perf_counter() - start) * 1000)

print("Average latency:", round(np.mean(times), 3), "ms")
print("Median latency:", round(np.median(times), 3), "ms")
print("95th percentile:", round(np.percentile(times, 95), 3), "ms")

Average latency: 3.438 ms
Median latency: 2.797 ms
95th percentile: 5.132 ms


In [ ]:
model_path = REPORT_DIR / "catboost_pipeline.joblib"

model_size_mb = model_path.stat().st_size / (1024 ** 2)

print("Model size:", round(model_size_mb, 2), "MB")

Model size: 0.56 MB


In [13]:
feature_audit = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [data[c].dtype for c in feature_cols],
    "missing_%": [
        data[c].isna().mean() * 100 for c in feature_cols
    ],
    "unique_values": [
        data[c].nunique() for c in feature_cols
    ]
})

display(
    feature_audit.sort_values("feature")
    .round(2)
    .reset_index(drop=True)
)

,feature,dtype,missing_%,unique_values
0,bin,int64,0.0,276
1,cam1_bw,float64,0.0,20
2,cam1_bw_last15,float64,0.0,46
3,cam1_fw,float64,0.0,19
4,cam1_fw_last15,float64,0.0,52
5,cam2_bw,float64,0.0,28
6,cam2_bw_last15,float64,0.0,54
7,cam2_fw,float64,0.0,24
8,cam2_fw_last15,float64,0.0,56
9,capacity,int64,0.0,2


In [14]:
print("BIN SUMMARY")
display(data["bin"].describe())

print("\nFirst 20 bin values:")
print(data["bin"].head(20).tolist())

print("\nUnique bin values:")
print(sorted(data["bin"].unique())[:30])

print("\nBIN BY ROOM")
display(
    data.groupby("room")["bin"]
    .agg(["min", "max", "nunique"])
)

print("\nWORKDAY VALUES")
print(data["workday"].value_counts(dropna=False))

BIN SUMMARY


count    36432.000000
mean       143.500000
std         79.674908
min          6.000000
25%         74.750000
50%        143.500000
75%        212.250000
max        281.000000
Name: bin, dtype: float64


First 20 bin values:
[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]

Unique bin values:
[np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35)]

BIN BY ROOM


,min,max,nunique
room,,,
1,6,281,276
2,6,281,276
3,6,281,276



WORKDAY VALUES
workday
1    36432
Name: count, dtype: int64


In [15]:
rit_feature_cols = [
    # Occupancy
    "occ",
    "occ_lag1",
    "occ_lag2",
    "occ_lag3",
    "occ_lag6",
    "occ_ratio",

    # CO2
    "co2",
    "co2_lag1",
    "co2_lag2",
    "co2_lag3",
    "co2_lag6",
    "co2_diff",
    "co2_age",

    # Temperature
    "temperature",
    "temperature_lag1",
    "temperature_lag2",
    "temperature_lag3",
    "temperature_lag6",
    "temperature_diff",
    "temperature_age",

    # Humidity
    "humidity",
    "humidity_lag1",
    "humidity_lag2",
    "humidity_lag3",
    "humidity_lag6",
    "humidity_diff",
    "humidity_age",

    # Illuminance
    "illuminance",
    "illuminance_lag1",
    "illuminance_lag2",
    "illuminance_lag3",
    "illuminance_lag6",
    "illuminance_age",

    # Time and room information
    "hour_sin",
    "hour_cos",
    "month",
    "room",
    "capacity",
    "is_lecture"
]

print("Original features:", len(feature_cols))
print("RIT-compatible candidates:", len(rit_feature_cols))

print("\nFeatures not found in data:")
print([c for c in rit_feature_cols if c not in data.columns])

print("\nFeatures selected:")
print(rit_feature_cols)

Original features: 55
RIT-compatible candidates: 39

Features not found in data:
[]

Features selected:
['occ', 'occ_lag1', 'occ_lag2', 'occ_lag3', 'occ_lag6', 'occ_ratio', 'co2', 'co2_lag1', 'co2_lag2', 'co2_lag3', 'co2_lag6', 'co2_diff', 'co2_age', 'temperature', 'temperature_lag1', 'temperature_lag2', 'temperature_lag3', 'temperature_lag6', 'temperature_diff', 'temperature_age', 'humidity', 'humidity_lag1', 'humidity_lag2', 'humidity_lag3', 'humidity_lag6', 'humidity_diff', 'humidity_age', 'illuminance', 'illuminance_lag1', 'illuminance_lag2', 'illuminance_lag3', 'illuminance_lag6', 'illuminance_age', 'hour_sin', 'hour_cos', 'month', 'room', 'capacity', 'is_lecture']


In [16]:
print("Checking occupancy ratio:")

ratio_check = (
    data["occ"] / data["capacity"]
)

print(
    "Maximum difference:",
    (data["occ_ratio"] - ratio_check).abs().max()
)

print("\nChecking time features:")
display(
    data[["bin", "hour_sin", "hour_cos"]].head(10)
)

print("\nChecking occupancy lags:")
display(
    data[["DayId", "room", "bin", "occ",
          "occ_lag1", "occ_lag2", "occ_lag3", "occ_lag6"]].head(10)
)

Checking occupancy ratio:
Maximum difference: 0.0

Checking time features:


,bin,hour_sin,hour_cos
0,6,0.130526,0.991445
1,7,0.152123,0.988362
2,8,0.173648,0.984808
3,9,0.195090,0.980785
4,10,0.216440,0.976296
5,11,0.237686,0.971342
6,12,0.258819,0.965926
7,13,0.279829,0.960050
8,14,0.300706,0.953717
9,15,0.321439,0.946930



Checking occupancy lags:


,DayId,room,bin,occ,occ_lag1,occ_lag2,occ_lag3,occ_lag6
0,0,1,6,0.0,0.0,0.0,0.0,0.0
1,0,1,7,0.0,0.0,0.0,0.0,0.0
2,0,1,8,0.0,0.0,0.0,0.0,0.0
3,0,1,9,0.0,0.0,0.0,0.0,0.0
4,0,1,10,0.0,0.0,0.0,0.0,0.0
5,0,1,11,0.0,0.0,0.0,0.0,0.0
6,0,1,12,0.0,0.0,0.0,0.0,0.0
7,0,1,13,0.0,0.0,0.0,0.0,0.0
8,0,1,14,0.0,0.0,0.0,0.0,0.0
9,0,1,15,0.0,0.0,0.0,0.0,0.0


In [17]:
# Keep the original feature list unchanged
X_rit = data[rit_feature_cols].copy()
y_rit = data["target_class"].astype(int).copy()

# Reuse the exact original train/test split
X_train_rit = X_rit.loc[X_train.index]
X_test_rit = X_rit.loc[X_test.index]

y_train_rit = y_rit.loc[y_train.index]
y_test_rit = y_rit.loc[y_test.index]

print("Training rows:", len(X_train_rit))
print("Testing rows:", len(X_test_rit))

print("Training features:", X_train_rit.shape[1])
print("Testing features:", X_test_rit.shape[1])

print("\nMissing training values:", X_train_rit.isna().sum().sum())
print("Missing testing values:", X_test_rit.isna().sum().sum())

print("\nTraining days:", groups.loc[X_train.index].nunique())
print("Testing days:", groups.loc[X_test.index].nunique())

Training rows: 28980
Testing rows: 7452
Training features: 39
Testing features: 39

Missing training values: 0
Missing testing values: 0

Training days: 35
Testing days: 9


In [18]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    balanced_accuracy_score
)
import time

rit_model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    random_seed=42,
    verbose=100
)

start_time = time.time()

rit_model.fit(
    X_train_rit,
    y_train_rit,
    eval_set=(X_test_rit, y_test_rit),
    early_stopping_rounds=100
)

training_time = time.time() - start_time

rit_predictions = rit_model.predict(X_test_rit).flatten().astype(int)

print("\nTraining time:", round(training_time, 2), "seconds")

print("\nClassification report:")
print(
    classification_report(
        y_test_rit,
        rit_predictions,
        target_names=CLASS_NAMES
    )
)

print("Accuracy:", round(
    accuracy_score(y_test_rit, rit_predictions), 4
))

print("Macro F1:", round(
    f1_score(y_test_rit, rit_predictions, average="macro"), 4
))

print("Weighted F1:", round(
    f1_score(y_test_rit, rit_predictions, average="weighted"), 4
))

print("Balanced accuracy:", round(
    balanced_accuracy_score(y_test_rit, rit_predictions), 4
))

0:	learn: 0.8043047	test: 0.7960834	best: 0.7960834 (0)	total: 28.5ms	remaining: 28.5s
100:	learn: 0.8392586	test: 0.8286305	best: 0.8302069 (91)	total: 3.31s	remaining: 29.5s
200:	learn: 0.8478578	test: 0.8300813	best: 0.8309224 (162)	total: 6.19s	remaining: 24.6s
300:	learn: 0.8563030	test: 0.8300571	best: 0.8322065 (241)	total: 7.84s	remaining: 18.2s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8322064832
bestIteration = 241

Shrink model to first 242 iterations.

Training time: 8.81 seconds

Classification report:
              precision    recall  f1-score   support

       Empty       0.92      0.91      0.92      3067
         Low       0.84      0.87      0.86      2874
    Moderate       0.67      0.66      0.67       761
        High       0.59      0.62      0.61       550
   Very high       0.65      0.35      0.46       200

    accuracy                           0.83      7452
   macro avg       0.73      0.68      0.70      7452
weighted avg      

In [22]:
import os
import json
import pandas as pd

# Create folders
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

# 1. Save trained model
rit_model.save_model(
    "../models/catboost_rit_39_features.cbm"
)

# 2. Save feature list
with open("../models/rit_feature_list.json", "w") as f:
    json.dump(rit_feature_cols, f, indent=4)

# 3. Save evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

metrics = {
    "Model": "CatBoost RIT-compatible",
    "Features": len(rit_feature_cols),
    "Training rows": len(X_train_rit),
    "Testing rows": len(X_test_rit),
    "Training time (seconds)": training_time,
    "Best iteration": rit_model.best_iteration_,
    "Accuracy": accuracy_score(y_test_rit, rit_predictions),
    "Macro F1": f1_score(
        y_test_rit, rit_predictions, average="macro"
    ),
    "Weighted F1": f1_score(
        y_test_rit, rit_predictions, average="weighted"
    ),
    "Balanced accuracy": balanced_accuracy_score(
        y_test_rit, rit_predictions
    )
}

pd.DataFrame([metrics]).to_csv(
    "../results/rit_model_metrics.csv", index=False
)

# 4. Save classification report
report = classification_report(
    y_test_rit,
    rit_predictions,
    target_names=CLASS_NAMES,
    output_dict=True
)

pd.DataFrame(report).transpose().to_csv(
    "../results/rit_classification_report.csv"
)

# 5. Save predictions
predictions_df = pd.DataFrame({
    "actual": y_test_rit.values,
    "predicted": rit_predictions
}, index=y_test_rit.index)

predictions_df.to_csv(
    "../results/rit_predictions.csv"
)

# 6. Save confusion matrix
cm = confusion_matrix(y_test_rit, rit_predictions)

pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv("../results/rit_confusion_matrix.csv")

print("Model and results saved successfully!")

Model and results saved successfully!


In [23]:
print(data[["target", "capacity"]].head())

print("\nTarget statistics:")
print((data["target"] / data["capacity"]).describe())

   target  capacity
0     0.0        84
1     0.0        84
2     0.0        84
3     0.0        84
4     0.0        84

Target statistics:
count    36432.000000
mean         0.118915
std          0.211237
min          0.000000
25%          0.000000
50%          0.012500
75%          0.137500
max          2.281250
dtype: float64


In [24]:
# Future occupancy ratio
y_reg = (data["target"] / data["capacity"]).copy()

# Use the same 39 deployment-compatible features
X_train_reg = X_rit.loc[X_train.index]
X_test_reg = X_rit.loc[X_test.index]

# Same train/test rows
y_train_reg = y_reg.loc[X_train.index]
y_test_reg = y_reg.loc[X_test.index]

print("Training rows:", len(X_train_reg))
print("Testing rows:", len(X_test_reg))

print("\nTarget distribution:")
print(y_train_reg.describe())

Training rows: 28980
Testing rows: 7452

Target distribution:
count    28980.000000
mean         0.114392
std          0.208234
min          0.000000
25%          0.000000
50%          0.011905
75%          0.125000
max          2.281250
dtype: float64


In [25]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import time

reg_model = CatBoostRegressor(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100
)

start_time = time.time()

reg_model.fit(
    X_train_reg,
    y_train_reg,
    eval_set=(X_test_reg, y_test_reg),
    early_stopping_rounds=100
)

reg_training_time = time.time() - start_time

reg_predictions = reg_model.predict(X_test_reg)

# Model evaluation
mae = mean_absolute_error(y_test_reg, reg_predictions)
rmse = np.sqrt(mean_squared_error(y_test_reg, reg_predictions))
r2 = r2_score(y_test_reg, reg_predictions)

# Persistence baseline: predict current occupancy ratio
baseline_predictions = data.loc[X_test_reg.index, "occ_ratio"]

baseline_mae = mean_absolute_error(y_test_reg, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test_reg, baseline_predictions))
baseline_r2 = r2_score(y_test_reg, baseline_predictions)

print("\nTraining time:", round(reg_training_time, 2), "seconds")

print("\nCatBoost Regression:")
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R²:", round(r2, 4))

print("\nPersistence Baseline:")
print("MAE:", round(baseline_mae, 4))
print("RMSE:", round(baseline_rmse, 4))
print("R²:", round(baseline_r2, 4))

0:	learn: 0.2003652	test: 0.2141540	best: 0.2141540 (0)	total: 8.22ms	remaining: 8.21s
100:	learn: 0.0772004	test: 0.0872444	best: 0.0872002 (98)	total: 770ms	remaining: 6.85s
200:	learn: 0.0703317	test: 0.0868628	best: 0.0867295 (191)	total: 2.09s	remaining: 8.31s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.08672951092
bestIteration = 191

Shrink model to first 192 iterations.

Training time: 3.25 seconds

CatBoost Regression:
MAE: 0.0423
RMSE: 0.0867
R²: 0.8469

Persistence Baseline:
MAE: 0.0406
RMSE: 0.0958
R²: 0.8132


In [26]:
reg_results = pd.DataFrame({
    "actual": y_test_reg,
    "predicted": reg_predictions
})

reg_results["actual_class"] = pd.cut(
    reg_results["actual"],
    bins=[-0.001, 0, 0.25, 0.50, 0.75, float("inf")],
    labels=CLASS_NAMES
)

reg_results["absolute_error"] = (
    reg_results["actual"] - reg_results["predicted"]
).abs()

display(
    reg_results.groupby(
        "actual_class",
        observed=False
    )["absolute_error"].agg(["count", "mean"])
)

,count,mean
actual_class,,
Empty,3067,0.008787
Low,2874,0.039028
Moderate,761,0.089123
High,550,0.127164
Very high,200,0.191534


In [27]:
import os

os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

# 1. Save trained regression model
reg_model.save_model(
    "../models/catboost_regression_39_features.cbm"
)

# 2. Save regression metrics
reg_metrics = {
    "Model": "CatBoost Regression",
    "Features": len(rit_feature_cols),
    "Training rows": len(X_train_reg),
    "Testing rows": len(X_test_reg),
    "Training time (seconds)": reg_training_time,
    "Best iteration": reg_model.best_iteration_,
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "Baseline MAE": baseline_mae,
    "Baseline RMSE": baseline_rmse,
    "Baseline R2": baseline_r2
}

pd.DataFrame([reg_metrics]).to_csv(
    "../results/regression_metrics.csv",
    index=False
)

# 3. Save predictions
reg_results.to_csv(
    "../results/regression_predictions.csv"
)

print("Regression model and results saved!")

Regression model and results saved!
